# Task 2: Setup, Evaluation Framework, and Baseline

Task 2 predicts one of four season labels—Fall, Spring, Summer, or Winter—from a catalogue image. Unlike article type, season is often a contextual or merchandising label rather than a directly observable visual property. The model must therefore infer season from indirect cues such as material, colour, garment coverage, and product type. Accessories and generic garments may plausibly belong to several seasons, which places an upper limit on what image evidence alone can explain.

The target is also strongly imbalanced: Summer represents approximately 49.3% of the training rows, whereas Spring represents only about 4.1%. A classifier that favours Summer can consequently obtain seemingly useful accuracy while performing poorly across the complete label set. This motivates the shared class-sensitive evaluation framework established below.

This notebook prepares Task 2 only. It creates the fixed season split, fits shared normalisation statistics, defines the evaluation metrics, evaluates non-learning baselines, and exports the common data contract used by every candidate model.

## How to Run

Execute the Task 2 notebooks in the following dependency order:

```text
01_task2_setup.ipynb
          |
          +----> 02_task2_random_forest.ipynb -----+
          +----> 03_task2_efficientnet_b0.ipynb ---+----> 05_task2_analysis_ultimate_judgement.ipynb
          +----> 04_task2_densenet121.ipynb -------+                         |
                                                                            v
                                                    06_task2_independent_evaluation.ipynb
                                                                            |
                                                                            v
                                                               07_task2_prediction.ipynb
```

1. **Prepare the shared evaluation contract:** run Notebook 01 first. It creates the only train-validation split used in Task 2, prepares aligned image and feature arrays, calculates training-only normalisation statistics, and exports baseline evidence.
2. **Train the candidate models:** run Notebooks 02, 03, and 04 after Notebook 01. They are independent of one another and may be executed in any order, but each must use the unchanged artefacts produced by the same Notebook 01 run.
3. **Compare and select:** run Notebook 05 only after all three model notebooks have produced their checkpoints. It validates their shared fingerprint and validation-row ordering, compares the candidates, selects the final model, and exports the analysis and durable selected-model package.
4. **Evaluate on separate external data:** run Notebook 06 after Notebook 05. It applies the selected package and the pinned external SigLIP model to the same labelled images from `ExtraSeasonData`, then compares Spring/Winter metrics, per-season correctness, off-label predictions, confusion matrices, and paired outcomes. This external comparison is kept separate from the frozen validation split and from local model selection.
5. **Generate final predictions:** run Notebook 07 last. It loads only the model selected in Notebook 05, predicts the unseen test rows in template order, and exports the final labels and score matrix. The external model is not used for final test prediction.

If the split, class order, image preprocessing, normalisation, or feature configuration changes here, rerun all three model notebooks before Notebook 05, followed by Notebooks 06 and 07. The saved fingerprint is designed to reject incompatible artefacts rather than combine results from different experimental runs.

## Required Structure to Run

Notebook 01 requires the repository marker, shared source modules, prepared training manifest, and original training images shown below. The manifest must contain the image identifiers/filenames and the `season` target used to locate matching files in `images_train`.

```text
Machine-Learning-Assignment-2/
├── pyproject.toml
├── src/
│   ├── preprocessing.py
│   └── task2_utils.py
├── preprocessed_datasets/
│   └── train_manifest.csv
└── datasets/
    └── train/
        └── images_train/
            └── <training image files>.jpg
```

The notebook creates `splits/task2_season_split.csv` and the complete `preprocessed_datasets/task2/` package consumed downstream: metadata, baseline scores, two image arrays, and two Random Forest feature arrays. Output directories are created automatically; the manifest and source images must already exist.

## 1. Setup


In [1]:
%matplotlib inline

import gc
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, densenet121

from joblib import Parallel, delayed, dump as joblib_dump, load as joblib_load
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0"]
MUTED = "#6b7280"


In [2]:
# Find the repository root whether Jupyter starts in the root or this notebook folder.
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    MANIFEST, TEST_IMAGE_DIR, IMAGE_TARGET_SIZE, compute_normalisation,
    describe_split, load_image_array, load_manifest, make_split,
)
from src.task2_utils import (
    FEATURE_CONFIG, extract_visual_features,
    TASK2_BASELINE_SCORES_PATH, TASK2_METADATA_PATH, TASK2_SPLIT_PATH,
    calculate_run_fingerprint, ensure_task2_directories,
)

ensure_task2_directories()
print("Repository root:", REPO_ROOT)
print("Manifest:", MANIFEST)
print("Image target size (w, h):", IMAGE_TARGET_SIZE)


Repository root: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2
Manifest: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\train_manifest.csv
Image target size (w, h): (60, 80)


### 1.1 Configuration

These settings control only the shared split and preprocessing. Model-specific training settings remain in each model notebook. Use the same `QUICK_RUN` value in Notebook 01 and every model notebook.


In [3]:
TARGET = "season"
RANDOM_STATE = 42
QUICK_RUN = False
VALIDATION_SHARE = 0.20
FEATURE_N_JOBS = -1

if QUICK_RUN:
    print("QUICK_RUN: preparing a reduced training cache for workflow testing.")
else:
    print("FULL RUN: preparing all Task 2 training rows.")


FULL RUN: preparing all Task 2 training rows.


## 2. Data and the Evaluation Framework

This section is fixed before model training. Every candidate is evaluated on the same validation rows with the same metrics.


### 2.1 Leakage-safe split

The shared `make_split` function keeps identical-image groups on one side of the split and stratifies by `season`. Classes with too few independent groups remain in training. The class mapping is fitted from training labels and saved with the final model.


In [4]:
frame = load_manifest(TARGET)
print(f"Rows carrying an {TARGET} label: {len(frame):,}")
print(f"Distinct classes in the manifest: {frame[TARGET].nunique()}")

if QUICK_RUN:
    # Reduce the eligible population before splitting while preserving its season distribution.
    # At 80:20, 6,250 eligible rows produce 5,000 training and 1,250 validation rows.
    quick_total = min(6250, len(frame))
    if quick_total < len(frame):
        _, frame = train_test_split(
            frame, test_size=quick_total, stratify=frame[TARGET],
            random_state=RANDOM_STATE,
        )
        frame = frame.reset_index(drop=True)
    print("QUICK_RUN: eligible population reduced to", len(frame))

train_frame, val_frame = make_split(
    frame, TARGET, validation_share=VALIDATION_SHARE, random_state=RANDOM_STATE
)

display(describe_split(train_frame, val_frame, TARGET))

Rows carrying an season label: 37,826
Distinct classes in the manifest: 4


,Target,Training rows,Validation rows,Achieved validation share %,Classes in training,Classes in validation,Classes absent from validation
0,season,30260,7566,20.002115,4,4,0


In [5]:
# Label encoding. Fixed to the sorted training classes and exported for all model notebooks, because
# reconstructing it later from a different frame would silently permute every prediction.
CLASSES = sorted(train_frame[TARGET].unique())

CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)

# Any validation class absent from training cannot be predicted. make_split sends
# single-group classes to training, so this should be empty; the check is what proves it.
unseen = sorted(set(val_frame[TARGET]) - set(CLASSES))
assert not unseen, f"Validation holds classes never seen in training: {unseen}"

y_train = train_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()
y_val = val_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()

train_support = pd.Series(np.bincount(y_train, minlength=N_CLASSES), index=CLASSES)
val_support = pd.Series(np.bincount(y_val, minlength=N_CLASSES), index=CLASSES)
SCOREABLE = np.flatnonzero(val_support.to_numpy() > 0)   # class indices macro averages use

print(f"Classes: {N_CLASSES}")
print(f"Scoreable in validation: {len(SCOREABLE)} | absent: {N_CLASSES - len(SCOREABLE)}")
print(f"Training support range: {train_support.max():,} down to {train_support.min()}")
print("Absent from validation:", sorted(np.array(CLASSES)[val_support.to_numpy() == 0]))

Classes: 4
Scoreable in validation: 4 | absent: 0
Training support range: 14,928 down to 1247
Absent from validation: []


In [6]:
class_table = pd.DataFrame({
    "Training images": train_support,
    "Validation images": val_support,
})
class_table["Training share %"] = class_table["Training images"] / len(y_train) * 100
display(class_table.style.format({"Training share %": "{:.1f}%"}))


,Training images,Validation images,Training share %
Fall,8208,2052,27.1%
Spring,1247,312,4.1%
Summer,14928,3732,49.3%
Winter,5877,1470,19.4%


**Finding.** Summer dominates both partitions, with 14,928 training and 3,732 validation images, whereas Spring has only 1,247 training and 312 validation images. The nearly identical class shares in the two partitions confirm that stratification preserved the target distribution. Nevertheless, Spring metrics are inherently less stable because each error represents a larger proportion of that class. Class-level results must therefore be interpreted together with their support rather than as equally precise estimates.


### 2.2 Loading images into memory

Images are standardised to 60×80 pixels using the deterministic transform established in Notebook 00. The transform preserves aspect ratio and fills unused space with the catalogue's white-background convention, avoiding the geometric distortion that direct stretching would introduce. This compact resolution substantially reduces memory use and training cost compared with conventional 224×224 CNN inputs, although some fine-grained texture information is inevitably lost.

Each image is decoded only once and retained as an unscaled `uint8` array. This keeps the reusable cache compact and prevents small numerical differences between model notebooks. Stochastic augmentation remains separate and is sampled during neural-network training, so caching does not freeze the augmented training distribution.


In [7]:
DEVICE = torch.device("cpu")  # setup does not need the GPU


In [8]:
def build_image_cache(frame, description):
    """Decode a frame's images once through the shared transform into one uint8 array.

    Only the deterministic transform from Section 3.1 of notebook 00 is applied here, exactly
    once per image. Augmentation still happens per epoch, so nothing about the training
    distribution is frozen by this cache.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8, in the frame's row order.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


def report_memory(label=""):
    """Host RSS and, on CUDA, device allocation. Cheap, and it makes a leak visible early."""
    line = []
    try:
        import resource
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        line.append(f"host peak {peak_kb / 1e6:.2f} GB")
    except (ImportError, AttributeError):
        try:
            import psutil
            line.append(f"host RSS {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        except ImportError:
            pass
    if DEVICE.type == "cuda":
        line.append(f"device allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB "
                    f"reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"[memory{' ' + label if label else ''}] " + " | ".join(line))


X_train_images = build_image_cache(train_frame, "train")
X_val_images = build_image_cache(val_frame, "validation")

assert len(X_train_images) == len(y_train) and len(X_val_images) == len(y_val)
report_memory("after caching")

  train: 10,000 / 30,260
  train: 20,000 / 30,260
  train: 30,000 / 30,260
train: 30,260 images in 136s (436 MB)
validation: 7,566 images in 6s (109 MB)
[memory after caching] host RSS 1.33 GB


### 2.3 Training-only normalisation

RGB mean and standard deviation are fitted on the Task 2 training rows only. The same constants are then applied to validation and test images.


In [9]:
VERIFY_NORMALISATION = False   # True: re-decode from disk and assert the constants match

start = time.time()
# Sums are taken over the raw 0-255 values in float32 and divided by 255 at the end, which is
# the same statistic as scaling first: sum(x/255) is sum(x)/255, and the same for the squares.
# The reductions accumulate into float64, so the running totals stay exact at this scale while
# the working chunk stays float32. Promoting the chunk itself to float64 would cost four bytes
# per channel per pixel twice over, once for the chunk and once for its square.
total = np.zeros(3, dtype=np.float64)
total_square = np.zeros(3, dtype=np.float64)
n_pixels = 0
for begin in range(0, len(X_train_images), 2048):
    chunk = X_train_images[begin:begin + 2048].astype(np.float32)
    total += chunk.sum(axis=(0, 1, 2), dtype=np.float64)
    total_square += np.einsum("nhwc,nhwc->c", chunk, chunk, dtype=np.float64)
    n_pixels += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]

mean_raw = total / n_pixels
variance_raw = np.maximum(total_square / n_pixels - mean_raw ** 2, 0.0)
NORM_MEAN = (mean_raw / 255.0).astype(np.float32)
NORM_STD = np.maximum(np.sqrt(variance_raw) / 255.0, 1e-6).astype(np.float32)

print(f"Fitted on {len(train_frame):,} training rows in {time.time() - start:.1f}s "
      "(from the cache, no second decode pass)")
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

if VERIFY_NORMALISATION:
    reference_mean, reference_std = compute_normalisation(train_frame,
                                                          target_size=IMAGE_TARGET_SIZE)
    print("Reference mean:", np.round(reference_mean, 4))
    print("Reference std: ", np.round(reference_std, 4))
    assert np.allclose(NORM_MEAN, reference_mean, atol=1e-4), "Mean disagrees with notebook 00"
    assert np.allclose(NORM_STD, reference_std, atol=1e-4), "Std disagrees with notebook 00"
    print("Verified against compute_normalisation.")

# White studio backgrounds dominate, so a mean near 0.9 is the expected result rather than a bug.
assert (NORM_MEAN > 0.5).all(), "Unexpectedly dark mean; check the transform before continuing."

Fitted on 30,260 training rows in 6.4s (from the cache, no second decode pass)
Mean (R, G, B): [0.8494 0.8325 0.8266]
Std  (R, G, B): [0.2717 0.2833 0.287 ]


### 2.4 Random Forest feature preprocessing

Random Forest requires a fixed numerical representation rather than a four-dimensional image tensor. The shared extractor therefore converts each prepared image into 2,007 descriptors from four complementary families:

- RGB and HSV summary statistics describe channel intensity, variation, and quartiles.
- Colour histograms describe the distribution of hue, saturation, and brightness.
- Histogram of Oriented Gradients (HOG) descriptors represent local edges, contours, and coarse shape.
- Foreground-occupancy measurements summarize how much of the frame is occupied and how that content is distributed vertically and horizontally.

Together, these features provide a reasonable classical-model comparison by exposing colour and shape cues that may correlate with season. They remain manually designed summaries, however, and cannot preserve every spatial relationship available to a CNN. The matrices are computed once and reused by Notebook 2 so that model fitting does not repeat image decoding or feature extraction.


In [10]:
def build_feature_cache(images, description):
    print(f"Extracting {description} visual features...")
    features = np.vstack(Parallel(n_jobs=FEATURE_N_JOBS)(
        delayed(extract_visual_features)(image) for image in images
    )).astype(np.float32)
    print(f"{description}: {features.shape[0]:,} rows x {features.shape[1]:,} features")
    return features

X_train_features = build_feature_cache(X_train_images, "training")
X_val_features = build_feature_cache(X_val_images, "validation")
assert len(X_train_features) == len(y_train) and len(X_val_features) == len(y_val)
print("Feature configuration:", FEATURE_CONFIG)


Extracting training visual features...
training: 30,260 rows x 2,007 features
Extracting validation visual features...
validation: 7,566 rows x 2,007 features
Feature configuration: {'hue_bins': 12, 'saturation_bins': 8, 'value_bins': 8, 'hog_orientations': 9, 'hog_pixels_per_cell': (8, 8), 'hog_cells_per_block': (2, 2), 'foreground_threshold': 0.95}


### 2.5 Metrics

Macro-F1 is the primary metric because every season should contribute equally. Accuracy, balanced accuracy and weighted F1 provide complementary views; balanced accuracy is the average recall across classes [1]. Top-2 accuracy measures whether the correct season appears among two suggestions.

In [11]:
def evaluate_predictions(y_true, y_pred, scores=None, name=""):
    labels = SCOREABLE
    row = {
        "Model": name,
        "Top-1 accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Balanced accuracy": recall_score(y_true, y_pred, labels=labels,
                                             average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if scores is not None:
        k = min(2, scores.shape[1])
        top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
        row["Top-2 accuracy"] = np.mean([truth in choices for truth, choices in zip(y_true, top_k)])
    else:
        row["Top-2 accuracy"] = np.nan
    return pd.DataFrame([row])


def per_class_table(y_true, y_pred):
    rows = []
    for index in SCOREABLE:
        true_binary = y_true == index
        pred_binary = y_pred == index
        rows.append({
            "Season": CLASSES[index],
            "Support": int(true_binary.sum()),
            "Precision": (true_binary & pred_binary).sum() / max(pred_binary.sum(), 1),
            "Recall": (true_binary & pred_binary).sum() / max(true_binary.sum(), 1),
            "F1": f1_score(true_binary, pred_binary, zero_division=0),
        })
    return pd.DataFrame(rows)


RESULTS = []

def record(result_frame):
    RESULTS.append(result_frame)
    display(result_frame.style.format({c: "{:.4f}" for c in result_frame.columns if c != "Model"}))
    return result_frame


### 2.6 Simple baselines

The majority baseline measures what class frequency alone can achieve. The stratified-random baseline samples from the training prior and provides a second non-learning reference. Random Forest is treated as a candidate model in Section 3, not as a trivial baseline.


In [12]:
majority_index = int(np.bincount(y_train, minlength=N_CLASSES).argmax())
majority_pred = np.full_like(y_val, majority_index)
majority_scores = np.zeros((len(y_val), N_CLASSES))
majority_scores[:, majority_index] = 1.0
print(f"Majority class: {CLASSES[majority_index]} "
      f"({train_support.iloc[majority_index]:,} training images)")
record(evaluate_predictions(y_val, majority_pred, majority_scores, "Baseline: majority class"))

prior = np.bincount(y_train, minlength=N_CLASSES) / len(y_train)
rng = np.random.RandomState(RANDOM_STATE)
stratified_pred = rng.choice(N_CLASSES, size=len(y_val), p=prior)
record(evaluate_predictions(
    y_val, stratified_pred, np.tile(prior, (len(y_val), 1)), "Baseline: stratified random"
))

Majority class: Summer (14,928 training images)


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: majority class,0.4933,0.1652,0.2500,0.3259,0.6875


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: stratified random,0.3533,0.2493,0.2493,0.3534,0.7645


,Model,Top-1 accuracy,Macro-F1,Balanced accuracy,Weighted F1,Top-2 accuracy
0,Baseline: stratified random,0.353291,0.249306,0.249325,0.353395,0.764473


**Finding.** The majority classifier reaches 49.3% accuracy by predicting Summer for every image, but its macro-F1 is only 0.165. The stratified-random baseline reaches 35.3% accuracy and 0.249 macro-F1. The disagreement between accuracy and macro-F1 demonstrates why overall accuracy alone would overstate performance on this imbalanced target. A useful trained model should improve substantially over both references and should show that the gain extends beyond the dominant Summer class.


## 3. Export Shared Task 2 Artefacts

The following artefacts form the reproducibility contract shared by the independent model notebooks. Keeping them together ensures that every candidate uses the same row ordering, labels, preprocessing statistics, and validation evidence.

| Artefact | Purpose |
|---|---|
| `splits/task2_season_split.csv` | Records each image ID, its fixed train or validation assignment, and its position within that partition. |
| `task2_metadata.json` | Stores class order, encoded labels, image size, normalisation statistics, feature configuration, row IDs, and the run fingerprint. |
| Deep-learning image arrays | Store the resized and padded `uint8` training and validation images in exact split order for both CNN notebooks. |
| Random-Forest feature arrays | Store the aligned 2,007-dimensional engineered features used by Notebook 2. |
| `task2_baseline_validation_scores.npz` | Preserves baseline scores, validation truth, class order, and validation IDs for the final comparison notebook. |

The prepared images are exported as NumPy `.npy` arrays so that JPEG decoding, aspect-ratio-preserving resizing, and white padding are performed once rather than repeated by every CNN notebook and every training run. A single array also preserves the exact row order associated with the saved labels and validation IDs, reducing the risk of image-label misalignment or small preprocessing differences between models. NumPy's memory-mapped loading allows downstream notebooks to access rows without first duplicating the complete dataset in memory. The cache remains `uint8` because it requires one byte per channel value, compared with four bytes for `float32`; scaling and training-only normalisation are applied later by the shared batch pipeline. This choice uses more disk space than compressed JPEG files but reduces repeated CPU decoding work and produces a faster, reproducible model input. Stochastic augmentation is still applied during training, so caching the deterministic image preparation does not freeze the augmented samples seen across epochs.

The run fingerprint is more than a convenient identifier: it binds the split, class order, preprocessing statistics, image geometry, and feature configuration into one compatibility check. Model checkpoints with a different fingerprint must not be compared or packaged together, because they do not represent the same experiment.


In [13]:
split_rows = pd.concat([
    pd.DataFrame({"id": train_frame["id"].astype(str), "split": "train",
                  "position": np.arange(len(train_frame))}),
    pd.DataFrame({"id": val_frame["id"].astype(str), "split": "validation",
                  "position": np.arange(len(val_frame))}),
], ignore_index=True)
TASK2_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_rows.to_csv(TASK2_SPLIT_PATH, index=False)

preprocessed_dir = REPO_ROOT / "preprocessed_datasets" / "task2"
preprocessed_dir.mkdir(parents=True, exist_ok=True)
def save_preprocessed_array(path, array, chunk_rows=1024):
    """Create, reuse, or atomically replace a prepared NumPy array.

    Model notebooks load these files as read-only memory maps. On Windows an open
    mapping cannot be truncated, so write the replacement beside the destination
    first and atomically swap it into place only after the write succeeds.
    """
    path = Path(path)
    if path.exists():
        existing = np.load(path, mmap_mode="r")
        compatible = existing.shape == array.shape and existing.dtype == array.dtype
        identical = compatible and all(
            np.array_equal(existing[start:start + chunk_rows], array[start:start + chunk_rows])
            for start in range(0, len(array), chunk_rows)
        )
        del existing
        if identical:
            print("Reused unchanged preprocessed array:", path.name)
            return
    existed = path.exists()
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    temporary.unlink(missing_ok=True)
    try:
        with temporary.open("xb") as handle:
            np.save(handle, array)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except OSError as error:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Could not replace prepared array {path}. Close any model notebook "
            "kernel that has this memory-mapped file open, then rerun this cell."
        ) from error
    print("Replaced preprocessed array:" if existed else "Saved preprocessed array:", path.name)

save_preprocessed_array(preprocessed_dir / "task2_deep_learning_train_images.npy", X_train_images)
save_preprocessed_array(preprocessed_dir / "task2_deep_learning_validation_images.npy", X_val_images)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_train_features.npy", X_train_features)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_validation_features.npy", X_val_features)

fingerprint = calculate_run_fingerprint(
    target=TARGET, classes=CLASSES, train_ids=train_frame["id"],
    validation_ids=val_frame["id"], random_state=RANDOM_STATE,
    image_target_size=IMAGE_TARGET_SIZE, normalisation_mean=NORM_MEAN,
    normalisation_std=NORM_STD, feature_version=FEATURE_CONFIG,
)
config = {
    "target": TARGET, "classes": CLASSES, "random_state": RANDOM_STATE, "quick_run": QUICK_RUN,
    "validation_share": VALIDATION_SHARE, "image_target_size": list(IMAGE_TARGET_SIZE),
    "normalisation_mean": NORM_MEAN.tolist(), "normalisation_std": NORM_STD.tolist(),
    "feature_config": FEATURE_CONFIG, "feature_count": int(X_train_features.shape[1]),
    "fingerprint": fingerprint,
    "train_ids": train_frame["id"].astype(str).tolist(),
    "validation_ids": val_frame["id"].astype(str).tolist(),
    "y_train": y_train.astype(int).tolist(),
    "y_validation": y_val.astype(int).tolist(),
}
TASK2_METADATA_PATH.write_text(json.dumps(config, indent=2))
with TASK2_BASELINE_SCORES_PATH.open("wb") as handle:
    np.savez_compressed(
        handle, fingerprint=fingerprint,
        validation_ids=val_frame["id"].astype(str).to_numpy(dtype=str),
        true_indices=y_val, classes=np.asarray(CLASSES),
        majority_scores=majority_scores,
    )
print("Saved split:", TASK2_SPLIT_PATH)
print("Run fingerprint:", fingerprint)
print("Saved metadata:", TASK2_METADATA_PATH)
print("Saved baseline evidence:", TASK2_BASELINE_SCORES_PATH)

print("Saved preprocessed arrays:", preprocessed_dir)


Replaced preprocessed array: task2_deep_learning_train_images.npy
Replaced preprocessed array: task2_deep_learning_validation_images.npy
Replaced preprocessed array: task2_random_forest_train_features.npy
Replaced preprocessed array: task2_random_forest_validation_features.npy
Saved split: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\splits\task2_season_split.csv
Run fingerprint: 5f507c7bb57f
Saved metadata: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2\task2_metadata.json
Saved baseline evidence: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2\task2_baseline_validation_scores.npz
Saved preprocessed arrays: D:\RMIT\Machine Learning\Machine-Learning-Assignment-2\preprocessed_datasets\task2


## References

[1] K. H. Brodersen, C. S. Ong, K. E. Stephan, and J. M. Buhmann, “The Balanced Accuracy and Its Posterior Distribution,” *20th International Conference on Pattern Recognition*, 2010. https://doi.org/10.1109/ICPR.2010.764
